# Modeling — **SVM + TF-IDF (sklearn)** (Colab)

Driver lengkap pendamping IndoBERT: clone repo → `train_svm_full14k` (GridSearch 24
kombinasi, split kanonik) → tampilkan SELURUH proses (grid + heatmap, confusion +
classification report, top fitur per kelas, contoh komentar + analisis error, keyakinan).
Pilih dataset di **bagian 2** (default **balanced3k**). CPU cukup.

> **SVM Spark MLlib** TIDAK di sini — jalur Big Data, dijalankan di cluster (`src/spark/`).

## 1. Clone repo (privat) + dependensi

In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (classic, scope repo): ')
# Clone bila belum ada; bila sudah ada, SYNC paksa ke versi terbaru (data+kode).
!if [ -d jokowi_sentiment_project ]; then cd jokowi_sentiment_project && git fetch -q origin && git reset -q --hard origin/main && cd ..; else git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git; fi
%cd jokowi_sentiment_project
%pip install -q 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv

## 2. Set MONGO_URI + jalankan training
`raw_comments` (label) + `processed_svm` (`svm` stemmed) dari Mongo Atlas. Subset → split 2100/600/300.

In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# === Pilih dataset ===
#   balanced 3k : TAG, SUBSET = 'balanced3k', 'outputs/labeling/balanced_3000.csv'
#   full 14k    : TAG, SUBSET = 'full14k', ''
TAG    = 'balanced3k'
SUBSET = 'outputs/labeling/balanced_3000.csv'

flags = f'--tag {TAG}' + (f' --subset {SUBSET}' if SUBSET else '')
!python -m src.modeling.train_svm_full14k $flags

## 3. Hasil grid search (24 kombinasi)
`ngram × min_df × C`, diurut peringkat macro-F1 validasi (`svm_<tag>_grid.csv`).

In [ ]:
import pandas as pd
g = pd.read_csv(f'outputs/reports/svm_{TAG}_grid.csv')
print(f'{len(g)} kombinasi. 8 teratas:'); display(g.head(8))

In [ ]:
import matplotlib.pyplot as plt
piv = g.pivot_table(index='param_clf__C', columns='param_tfidf__min_df', values='mean_test_score', aggfunc='max')
fig, ax = plt.subplots(figsize=(5.5, 4)); im = ax.imshow(piv.values, cmap='viridis')
ax.set_xticks(range(len(piv.columns)), piv.columns); ax.set_yticks(range(len(piv.index)), piv.index)
ax.set_xlabel('min_df'); ax.set_ylabel('C'); ax.set_title('Grid search macro-F1 (maks lintas ngram)')
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f'{piv.values[i,j]:.3f}', ha='center', va='center', color='white')
fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

## 4. Evaluasi test: confusion matrix + classification report

In [ ]:
import json, numpy as np
from IPython.display import Image, display
m = json.load(open(f'outputs/reports/svm_{TAG}_metrics.json'))['test']
print(f"Akurasi : {m['accuracy']}  | macro-F1: {m['macro_f1']}  | best params: {m.get('best_params')}")
display(Image(f'outputs/reports/svm_{TAG}_confusion.png'))

In [ ]:
import numpy as np, pandas as pd
L = ['Negatif', 'Netral', 'Positif']
cm = np.array(m['confusion_matrix'])
rows = []
for i, l in enumerate(L):
    tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    rows.append({'kelas': l, 'precision': round(p, 3), 'recall': round(r, 3),
                 'f1': round(f, 3), 'support': int(cm[i].sum())})
rep = pd.DataFrame(rows)
print('Classification report (dihitung dari confusion matrix):')
display(rep)

In [ ]:
import matplotlib.pyplot as plt
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5, 4.3)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(3), L); ax.set_yticks(range(3), L); ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
ax.set_title('Confusion ternormalisasi (recall per baris)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cmn[i,j]:.2f}', ha='center', va='center', color='white' if cmn[i, j] > .5 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

In [ ]:
x = np.arange(3); w = 0.25
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(x - w, rep['precision'], w, label='precision')
ax.bar(x,     rep['recall'],    w, label='recall')
ax.bar(x + w, rep['f1'],        w, label='F1')
ax.set_xticks(x, L); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=.3)
ax.set_title('Precision / Recall / F1 per kelas'); plt.tight_layout(); plt.show()

## 5. Top fitur diskriminatif per kelas
Kata/bigram dengan koefisien LinearSVC tertinggi tiap kelas (interpretabilitas).

In [ ]:
tf = pd.read_csv(f'outputs/reports/svm_{TAG}_top_features.csv')
tab = {lab: tf[tf.kelas == lab].sort_values('rank')['fitur'].head(10).tolist()
       for lab in ['Negatif', 'Netral', 'Positif']}
display(pd.DataFrame(tab))

## 6. Contoh komentar + klasifikasi (+ analisis error)
Test set (`svm_<tag>_predictions.csv`). ❌ = salah klasifikasi.

In [ ]:
import pandas as pd
p = pd.read_csv(f'outputs/reports/svm_{TAG}_predictions.csv')
print(f'Test: {len(p)} komentar | benar {p["benar"].sum()} | salah {(~p["benar"]).sum()} | akurasi {p["benar"].mean():.4f}')
def tampil(df, n):
    for _, r in df.head(n).iterrows():
        mark = '\u2705' if r['benar'] else '\u274c'
        t = (str(r['text'])[:110]).replace(chr(10), ' ')
        print(f"{mark} asli={r['label_asli']:<7} pred={r['prediksi']:<7} | {t}")
print(chr(10) + '\u2014 Contoh SALAH \u2014')
tampil(p[~p['benar']], 12)
print(chr(10) + '\u2014 Contoh BENAR \u2014')
tampil(p[p['benar']], 8)
p['teks'] = p['text'].astype(str).str.slice(0, 90)
print(chr(10) + 'Tabel (20 pertama):')
display(p[['label_asli', 'prediksi', 'benar', 'keyakinan', 'teks']].head(20))

## 7. Analisis keyakinan
Prediksi salah dengan margin keputusan tertinggi = model paling yakin tapi salah
(kandidat **label LLM keliru** / jebakan leksikal).

In [ ]:
import matplotlib.pyplot as plt
p = pd.read_csv(f'outputs/reports/svm_{TAG}_predictions.csv')
if 'keyakinan' in p.columns:
    err = p[~p['benar']].sort_values('keyakinan', ascending=False)
    print(f'Prediksi SALAH paling YAKIN ({len(err)} salah dari {len(p)}) — kandidat label LLM keliru:')
    for _, r in err.head(10).iterrows():
        t = (str(r['text'])[:95]).replace(chr(10), ' ')
        print(f"  k={r['keyakinan']:.3f} asli={r['label_asli']:<7} pred={r['prediksi']:<7} | {t}")
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.hist(p[p['benar']]['keyakinan'], bins=20, alpha=.6, label='benar')
    ax.hist(p[~p['benar']]['keyakinan'], bins=20, alpha=.6, label='salah')
    ax.set_xlabel('keyakinan model'); ax.set_ylabel('jumlah'); ax.legend()
    ax.set_title('Distribusi keyakinan: benar vs salah'); plt.tight_layout(); plt.show()
else:
    print('Kolom keyakinan belum ada — jalankan ulang training versi terbaru.')

## 8. (Opsional) Bandingkan 3 model

In [ ]:
import os, numpy as np, pandas as pd
from IPython.display import Image, display
suffix = '' if TAG == 'full14k' else f'_{TAG}'
need = [f'outputs/reports/svm_{TAG}_metrics.json',
        f'outputs/reports/svm_spark{suffix}_metrics.json',
        f'outputs/reports/indobert{suffix}_metrics.json']
if all(os.path.exists(x) for x in need):
    !python -m src.modeling.compare_models --tag $TAG
    csv_name = 'model_comparison_full14k.csv' if TAG == 'full14k' else f'model_comparison_{TAG}.csv'
    png_name = 'model_comparison_accuracy.png' if TAG == 'full14k' else f'model_comparison_{TAG}_accuracy.png'
    cmp = pd.read_csv(f'outputs/reports/{csv_name}'); display(cmp)
    x = np.arange(3); w = 0.25; cols = ['f1_negatif', 'f1_netral', 'f1_positif']
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for k, (_, row) in enumerate(cmp.iterrows()):
        ax.bar(x + (k - 1) * w, [row[c] for c in cols], w, label=row['model'])
    ax.set_xticks(x, ['Negatif', 'Netral', 'Positif']); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=.3)
    ax.set_title(f'F1 per kelas — 3 model ({TAG})'); plt.tight_layout(); plt.show()
    display(Image(f'outputs/reports/{png_name}'))
else:
    print('Lewati: butuh ketiga JSON. Ada:', [x for x in need if os.path.exists(x)])

In [ ]:
from google.colab import files
for f in [f'svm_{TAG}_metrics.json', f'svm_{TAG}_confusion.png', f'svm_{TAG}_grid.csv',
          f'svm_{TAG}_top_features.csv', f'svm_{TAG}_predictions.csv']:
    files.download(f'outputs/reports/{f}')